In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("event_type", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("city", StringType(), True)
])

In [0]:
import requests
import json
response= requests.get("https://holmes-hampton-going-busy.trycloudflare.com/event")
# # url="https://holmes-hampton-going-busy.trycloudflare.com/events?count=10"
data = response.json()
with open("/Volumes/file_upload/files/tmp/delta/user_event_checkpoint/raw_landing/data.json", "w") as f:
    json.dump(data,f)
print(response.status_code)
print(response.json())

In [0]:
bronze_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format","json") \
    .option("cloudFiles.schemaLocation","/Volumes/file_upload/files/tmp/delta/bronze_event_checkpoint/bronze_schema") \
    .load("/Volumes/file_upload/files/tmp/delta/user_event_checkpoint/raw_landing/")

In [0]:
bronze_df.writeStream \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/file_upload/files/tmp/delta/bronze_event_checkpoint/") \
    .trigger(availableNow=True) \
    .toTable("ecommerce.bronze.bronze_events")